# LFP dataset: Complete and segmented discharge curves

Adapted from the supplied dataset-visualization workflow. This notebook uses the
existing MAT files in `../../dataset`; no duplicate dataset copy is needed.
Figures: `dataset_visualization/outputs/complete-curves`. Numerical source tables and
input hashes: `dataset_visualization/results/complete-curves`.
Run all cells in order. See the parent README for definitions and interpretation.


In [ ]:
from pathlib import Path
import sys

# Resolve from a notebook subfolder, the LFP module, or the release root.
roots = (Path.cwd(), *Path.cwd().parents)
candidates = [p for root in roots for p in
              (root, root / "dataset_visualization", root / "LFP dataset/dataset_visualization")]
PROJECT_ROOT = next(p for p in candidates if (p / "notebook_setup.py").is_file())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from notebook_setup import get_dataset_path, save_figure, save_table, finish_category, begin_category
CATEGORY = 'complete-curves'
begin_category(CATEGORY)


import scipy.io as sio
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.cm import ScalarMappable

# ============================================================
# 1. Load data
# ============================================================
mat_path = get_dataset_path("LFP_data_processed.mat")
data = sio.loadmat(mat_path)

processed_Qd_mat   = data['processed_Qd_mat']     # 201 x 3511
processed_Vd_mat   = data['processed_Vd_mat']     # 201 x 3511
process_qIC_mat    = data['processed_qIC_mat']    # 200 x 3511
processed_vIC_mat  = data['processed_vIC_mat']    # 200 x 3511

# Maximum capacity of each cell/sample
Qmax_all = processed_Qd_mat[-1, :]                # 3511,

# ============================================================
# 2. Nature-style global settings
# ============================================================
plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 16,
    "axes.linewidth": 1.2,
    "axes.labelsize": 16,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "xtick.major.width": 1.1,
    "ytick.major.width": 1.1,
    "xtick.major.size": 4,
    "ytick.major.size": 4,
    "legend.frameon": False,
    "figure.dpi": 300
})

# ============================================================
# 3. Nature-like low-saturation colormap
# ============================================================
nature_colors = [
    "#3B5B92",   # muted blue
    "#6FA8A3",   # muted cyan-green
    "#D9C77E",   # muted yellow
    "#D8905A",   # muted orange
    "#B85C5C"    # muted red
]

cmap = LinearSegmentedColormap.from_list(
    "nature_capacity_cmap",
    nature_colors,
    N=256
)

norm = Normalize(
    vmin=np.nanmin(Qmax_all),
    vmax=np.nanmax(Qmax_all)
)

# ============================================================
# 4. Create 1 x 2 figure layout
# ============================================================
fig, axes = plt.subplots(
    1, 2,
    figsize=(12, 4.8),
    dpi=300,
    constrained_layout=False
)

ax1, ax2 = axes

# ============================================================
# Add subplot labels
# ============================================================
ax1.text(
    -0.16, 1.05, "a",
    transform=ax1.transAxes,
    fontsize=18,
    fontweight="bold",
    va="top",
    ha="left"
)

ax2.text(
    -0.16, 1.05, "b",
    transform=ax2.transAxes,
    fontsize=18,
    fontweight="bold",
    va="top",
    ha="left"
)

n_samples = processed_Qd_mat.shape[1]

# ============================================================
# 5. Left panel: V-Q curves
#    x-axis: capacity Q
#    y-axis: voltage V
# ============================================================
for i in range(n_samples):
    color = cmap(norm(Qmax_all[i]))

    ax1.plot(
        processed_Qd_mat[:, i],
        processed_Vd_mat[:, i],
        color=color,
        linewidth=0.8,
        alpha=0.55
    )

ax1.set_xlabel("Discharge capacity (Ah)", labelpad=10)
ax1.set_ylabel("Voltage (V)", labelpad=10)

ax1.grid(False)
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)

# ============================================================
# 6. Right panel: IC curves
#    x-axis: voltage
#    y-axis: dQ/dV
# ============================================================
n_samples_ic = process_qIC_mat.shape[1]

for i in range(n_samples_ic):
    color = cmap(norm(Qmax_all[i]))

    ax2.plot(
        processed_vIC_mat[:, i],
        process_qIC_mat[:, i],
        color=color,
        linewidth=0.8,
        alpha=0.55
    )

ax2.set_xlabel("Voltage (V)", labelpad=10)
ax2.set_ylabel("dQ/dV (Ah·V$^{-1}$)", labelpad=10)

ax2.grid(False)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)

# ============================================================
# 7. Shared colorbar
# ============================================================
sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])

# Adjust subplot spacing and reserve space for colorbar
fig.subplots_adjust(
    left=0.08,
    right=0.88,
    bottom=0.18,
    top=0.95,
    wspace=0.32
)

# Dedicated colorbar axis
cbar_ax = fig.add_axes([0.91, 0.22, 0.018, 0.65])
cbar = fig.colorbar(sm, cax=cbar_ax)

cbar.set_label("Maximum capacity (Ah)", labelpad=10)

# Remove colorbar border
cbar.outline.set_visible(False)

# Optional: remove colorbar tick border effect
cbar.ax.tick_params(
    direction="out",
    width=1.0,
    length=3
)

# ============================================================
# 8. Save figure
# ============================================================
save_figure(fig, CATEGORY, 
    "VQ_and_IC_curves_shared_colorbar_Nature_style.png",
    dpi=300,
    bbox_inches="tight"
)

pass  # Export formats are selected by the shared save helper.

# Figures are saved below; no GUI window is required.
save_table(CATEGORY, 'curve_columns.csv', cell_index_zero_based=np.arange(n_samples), capacity_Ah=Qmax_all)

finish_category(CATEGORY)
